In [1]:
import pandas as pd
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

scraped_df = pd.read_csv("../data/interim/scraped_stackexchange_stress_combined.csv")

print("Shape:", scraped_df.shape)
print("\nColumns:")
print(scraped_df.columns.tolist())

print("\nClass counts:")
print(scraped_df["stress_level"].value_counts())

scraped_df.head()

Shape: (1240, 15)

Columns:
['id', 'text', 'stress_level', 'source_type', 'source_name', 'source_url', 'source_record_id', 'original_label', 'label_mapping_rule', 'language', 'date_collected', 'collector', 'labeler', 'review_status', 'notes']

Class counts:
stress_level
Low       493
Medium    393
High      354
Name: count, dtype: int64


,id,text,stress_level,source_type,source_name,source_url,source_record_id,original_label,label_mapping_rule,language,date_collected,collector,labeler,review_status,notes
0,scraped_combined_00001,Is it okay to publish my thesis when it was do...,Medium,online_scraping_api,Stack Exchange API,https://academia.stackexchange.com/questions/1...,107623,academia:concerned about grades,keyword_query_mapping_v1,English,2026-04-27,project_team,keyword_mapping_needs_manual_review,needs_review,Scraped online text; verify label manually bef...
1,scraped_combined_00002,Avoiding conferences: Potential effects on an ...,Low,online_scraping_api,Stack Exchange API,https://academia.stackexchange.com/questions/4...,49438,academia:good time management,keyword_query_mapping_v1,English,2026-04-27,project_team,keyword_mapping_needs_manual_review,needs_review,Scraped online text; verify label manually bef...
2,scraped_combined_00003,Should I have said anything to my team member ...,Low,online_scraping_api,Stack Exchange API,https://workplace.stackexchange.com/questions/...,198713,workplace:good time management,keyword_query_mapping_v1,English,2026-04-27,project_team,keyword_mapping_needs_manual_review,needs_review,Scraped online text; verify label manually bef...
3,scraped_combined_00004,How to deal with a manager who deliberately ov...,Medium,online_scraping_api,Stack Exchange API,https://workplace.stackexchange.com/questions/...,149896,workplace:stress workload manage,keyword_query_mapping_v1,English,2026-04-27,project_team,keyword_mapping_needs_manual_review,needs_review,Scraped online text; verify label manually bef...
4,scraped_combined_00005,Why do many talented scientists write horrible...,Low,online_scraping_api,Stack Exchange API,https://academia.stackexchange.com/questions/1...,17781,academia:good time management,keyword_query_mapping_v1,English,2026-04-27,project_team,keyword_mapping_needs_manual_review,needs_review,Scraped online text; verify label manually bef...


In [2]:
print("Missing text values:", scraped_df["text"].isna().sum())
print("Duplicate text values:", scraped_df["text"].duplicated().sum())

scraped_df["word_count"] = scraped_df["text"].astype(str).apply(lambda x: len(x.split()))

print("\nWord count statistics:")
print(scraped_df["word_count"].describe())

Missing text values: 0
Duplicate text values: 0

Word count statistics:
count    1240.000000
mean      372.607258
std       217.078354
min        44.000000
25%       223.750000
50%       348.500000
75%       500.000000
max      2643.000000
Name: word_count, dtype: float64


In [3]:
clean_scraped_df = scraped_df.copy()

# Remove missing text
clean_scraped_df = clean_scraped_df.dropna(subset=["text"])

# Remove duplicate text
clean_scraped_df = clean_scraped_df.drop_duplicates(subset=["text"])

# Recalculate word count
clean_scraped_df["word_count"] = clean_scraped_df["text"].astype(str).apply(lambda x: len(x.split()))

# Remove very short text
clean_scraped_df = clean_scraped_df[clean_scraped_df["word_count"] >= 20]

# Remove extremely long text to keep model training cleaner
clean_scraped_df = clean_scraped_df[clean_scraped_df["word_count"] <= 500]

print("Shape after cleaning:", clean_scraped_df.shape)
print("\nClass counts after cleaning:")
print(clean_scraped_df["stress_level"].value_counts())

Shape after cleaning: (1118, 16)

Class counts after cleaning:
stress_level
Low       410
Medium    358
High      350
Name: count, dtype: int64


In [4]:
pd.set_option("display.max_colwidth", 300)

clean_scraped_df[["text", "stress_level", "original_label", "source_url"]].sample(
    10, random_state=42
)

,text,stress_level,original_label,source_url
1118,Is it rude to leave and switch to a lab that's collaborating with your own lab (and how to go about doing so)?. I've signaled to my advisor and collaborators my intent to quit pursuing my PhD at the school I'm currently at. I essentially gave a 3-month notice while I get as much done as I can on...,Medium,academia:busy schedule,https://academia.stackexchange.com/questions/150335/is-it-rude-to-leave-and-switch-to-a-lab-thats-collaborating-with-your-own-lab
187,Boss is wilfully forgetful. My boss seems to suffer from wilful forgetfulness. I manage a small team who have been recently given an increased workload. My boss told me that he would add a new member to our team to allow us to handle the new work. I had made him aware that were already spread ve...,Low,workplace:balanced workload,https://workplace.stackexchange.com/questions/105990/boss-is-wilfully-forgetful
608,"Changing fields during PhD. I’m three months into a PhD (in the UK) at a professional school exploring AI Ethics. My undergraduate background is in Philosophy and I jumped directly into my PhD (~3 years with no taught elements). I’ve not done any work on AI before, and am feeling quite overwhelm...",High,academia:overwhelmed,https://academia.stackexchange.com/questions/215788/changing-fields-during-phd
833,"How can an individual stop the university's standards and quality from deteriorating?. The business school I am currently at faces a very difficult situation. After a scandal involving fraud around the president a few years ago, in combination probably with fallout from the financial crisis and ...",Medium,academia:time management problem,https://academia.stackexchange.com/questions/62692/how-can-an-individual-stop-the-universitys-standards-and-quality-from-deteriora
899,"How can you make a stronger PhD application by going to masters program? (Mathematics). I am an undergraduate student studying mathematics. My undergraduate school had a very small math department. My school didn’t offer as many courses as other big universities, and it didn’t offer research opp...",Low,academia:not stressed,https://academia.stackexchange.com/questions/217925/how-can-you-make-a-stronger-phd-application-by-going-to-masters-program-mathem
915,How do you balance school and study?. I know there's many different questions that involve having a life with a healthy work-life balance but my question is different: I'm looking for a balance between my school life(high school) and studying. But I don't mean studying for school-related stuff. ...,Low,academia:work life balance,https://academia.stackexchange.com/questions/139006/how-do-you-balance-school-and-study
647,"Does prestige matter for a masters thesis? Need help to choose a thesis. I'm a master student in a very good university in the EU majoring in biomedicine/biology. I'm leaning towards doing a PhD after my masters, but either way eventually I will go into industry (whether I do a PhD or not). Soon...",Low,academia:not stressed,https://academia.stackexchange.com/questions/201856/does-prestige-matter-for-a-masters-thesis-need-help-to-choose-a-thesis
818,"How to tell professor the reason for dropping his/her subject without it sounding rude?. In our university, we are allowed to drop or cancel our classes before a certain deadline within the semester. The reason for dropping is, officially, not required but most professors I know usually asks the...",Medium,academia:worried deadline,https://academia.stackexchange.com/questions/126090/how-to-tell-professor-the-reason-for-dropping-his-her-subject-without-it-soundin
232,"Was I wrong to be candid with a potential new hire about my reasons for leaving the company?. A little bit of backstory: I quit my job 2 months ago and I am serving my notice period. I had some issues with the management because I did not accept their new exploitative ""leadership"" and they start...",Low,workplace:good time management,https://workplace.stackexchang

In [5]:
clean_scraped_df = clean_scraped_df.drop(columns=["word_count"], errors="ignore")

clean_scraped_df.to_csv("../data/processed/scraped_stackexchange_stress_clean.csv", index=False)

print("Saved cleaned scraped dataset:")
print("../data/processed/scraped_stackexchange_stress_clean.csv")

print("\nFinal cleaned scraped counts:")
print(clean_scraped_df["stress_level"].value_counts())

Saved cleaned scraped dataset:
scraped_stackexchange_stress_clean.csv

Final cleaned scraped counts:
stress_level
Low       410
Medium    358
High      350
Name: count, dtype: int64


In [6]:
import pandas as pd
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

main_df = pd.read_csv("../data/interim/stress_level_dataset_v1.csv")
scraped_clean_df = pd.read_csv("../data/processed/scraped_stackexchange_stress_clean.csv")

print("Main dataset shape:", main_df.shape)
print("Scraped dataset shape:", scraped_clean_df.shape)

print("\nMain dataset counts:")
print(main_df["stress_level"].value_counts())

print("\nScraped dataset counts:")
print(scraped_clean_df["stress_level"].value_counts())

Main dataset shape: (4500, 15)
Scraped dataset shape: (1118, 15)

Main dataset counts:
stress_level
Low       1500
Medium    1500
High      1500
Name: count, dtype: int64

Scraped dataset counts:
stress_level
Low       410
Medium    358
High      350
Name: count, dtype: int64


In [7]:
combined_df = pd.concat([main_df, scraped_clean_df], ignore_index=True)

# Remove exact duplicate text if any exists between both datasets
combined_df["text_normalized"] = combined_df["text"].astype(str).str.lower().str.strip()
combined_df = combined_df.drop_duplicates(subset=["text_normalized"])
combined_df = combined_df.drop(columns=["text_normalized"])

# Recreate clean IDs
combined_df["id"] = [f"combined_{i:05d}" for i in range(1, len(combined_df) + 1)]

print("Combined dataset shape:", combined_df.shape)
print("\nCombined class counts:")
print(combined_df["stress_level"].value_counts())

combined_df.to_csv("../data/processed/stress_level_dataset_v2_combined.csv", index=False)

print("\nSaved: stress_level_dataset_v2_combined.csv")

Combined dataset shape: (5594, 15)

Combined class counts:
stress_level
Low       1907
High      1846
Medium    1841
Name: count, dtype: int64

Saved: stress_level_dataset_v2_combined.csv


In [9]:
import pandas as pd
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

combined_df = pd.read_csv("../data/processed/stress_level_dataset_v2_combined.csv")

print("Combined dataset shape:", combined_df.shape)
print("\nCombined class counts:")
print(combined_df["stress_level"].value_counts())

min_class_count = combined_df["stress_level"].value_counts().min()

balanced_parts = []

for label in combined_df["stress_level"].unique():
    class_df = combined_df[combined_df["stress_level"] == label]
    sampled_class_df = class_df.sample(min_class_count, random_state=42)
    balanced_parts.append(sampled_class_df)

balanced_df = pd.concat(balanced_parts, ignore_index=True)

# Shuffle rows
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Recreate clean IDs
balanced_df["id"] = [f"balanced_{i:05d}" for i in range(1, len(balanced_df) + 1)]

print("\nBalanced dataset shape:", balanced_df.shape)
print("\nBalanced class counts:")
print(balanced_df["stress_level"].value_counts())

balanced_df.to_csv("../data/processed/stress_level_dataset_v2_balanced.csv", index=False)

print("\nSaved: stress_level_dataset_v2_balanced.csv")

Combined dataset shape: (5594, 15)

Combined class counts:
stress_level
Low       1907
High      1846
Medium    1841
Name: count, dtype: int64

Balanced dataset shape: (5523, 15)

Balanced class counts:
stress_level
Low       1841
High      1841
Medium    1841
Name: count, dtype: int64

Saved: stress_level_dataset_v2_balanced.csv


In [10]:
import pandas as pd
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

final_df = pd.read_csv("../data/processed/stress_level_dataset_v2_balanced.csv")

print("Final dataset shape:", final_df.shape)
print("\nFinal class counts:")
print(final_df["stress_level"].value_counts())

print("\nMissing values:")
print(final_df[["text", "stress_level"]].isna().sum())

print("\nDuplicate texts:")
print(final_df["text"].duplicated().sum())

final_df[["text", "stress_level", "source_type", "source_name"]].sample(10, random_state=42)

Final dataset shape: (5523, 15)

Final class counts:
stress_level
Low       1841
High      1841
Medium    1841
Name: count, dtype: int64

Missing values:
text            0
stress_level    0
dtype: int64

Duplicate texts:
0


,text,stress_level,source_type,source_name
1618,"Daily bloody noses, bloody sputum, dry blood in nose, just want someone to clear things up for me. I have bad winter allergies, and recently for the past week my nose has been very dry, i decided to turn on my phone flashlight and shine it in my nose just to see what it looked like. It appears m...",High,Mendeley Data public dataset,MentalDistress
4216,"you seem to be taking their comment the wrong way and you shouldn’t. no one likes to be corrected or told they’re wrong, but in this case it’s really not about you. there are a lot of misconceptions about abusive and unhealthy relationships, who gets into them, and the reasons they stay in them ...",Medium,Mendeley Data public dataset,MentalDistress
1964,"yeah i’m so bloody sick of christmas by the time december rolls around, just get it over with. i don’t even hate the holiday, i grew up celebrating it, but my god i can’t even get my damn groceries without hearing awful renditions of the same five songs over and over again. i’m just beyond sick ...",Medium,Mendeley Data public dataset,MentalDistress
5198,"Depends on where you live (the UK has Glisten Cosmetics, VE Cosmetics & Made by Mitchell, while the US has LA. Girl, RCMA cream foundation and Haus Labs, plus theatrical brands avaliable to both.) But the key thing you really want is a pure white setting powder, like Ben Nye's Super White that a...",Medium,Mendeley Data public dataset,MentalDistress
2100,"being 'needed' rather than 'wanted' more than happy to have her cry on my shoulder or demand cuddles 24/7. but if she can't look after herself and needs babying all the time, then i feel like i'm dating a child rather than a woman.",Low,Mendeley Data public dataset,MentalDistress
3428,"Are there specific organizational processes or tools (not necessarily software) that can help in developing and structuring research ideas?. I am doing research in computer Science as a PhD student. I have some research ideas and for months I've been continuously thinking about them, developing ...",Low,online_scraping_api,Stack Exchange API
4582,"""just google it"" is dead. the AI slop means the 'google fu' we all had 5 years ago is fuackall worthless. there is no more ""search the internet"" now thats a recipe for 3 hours down the tubes trying to figure out if the AI gave you correct info and you're an idiot, or if it gave you instructions ...",Medium,Mendeley Data public dataset,MentalDistress
4016,"Fear of colon cancer Hello. This might be a bit TMI but I need help here. Earlier this week a had developed a hemorrhoid on the outside of my anus. It pretty much covers the hole. I wasn’t really paying attention to my feces too much but for the past few days I have been pooping very thin, flat ...",High,Mendeley Data public dataset,MentalDistress
2462,"How important are your module choices in 3rd year mathematics undergraduate?. I am finishing my 2nd year doing a Mathematics BSc (UK) and I am very worried about my options next year. I primarily enjoy analysis and am interested in pure maths generally. However, my university, for various reason...",Medium,online_scraping_api,Stack Exchange API
3433,"Consequences for not attending an academic conference/forum in which your paper was already accepted?. BACKGROUND I was previously an engineering PhD student but I left the program. I was really too inexperienced for the commitment of a PhD. Further, my wife wanted me to leave and pursue an engi...",Medium,online_scraping_api,Stack Exchange API


In [ ]:
okay now before going to this step, I want you to explain to me what we have did in details all along since the start of project